In [1]:
import time
from datetime import datetime

import kubeflow
import kubeflow.trainer
from kubeflow.trainer.options import TrainerCommand
from kubeflow.trainer.types.types import CustomTrainerContainer

In [2]:
# Verification of Kubeflow dependency and their version
print(f"Kubeflow version: {kubeflow.__version__ if hasattr(kubeflow, '__version__') else 'N/A'}")

Kubeflow version: 0.3.0


In [3]:
config = kubeflow.trainer.KubernetesBackendConfig()
trainer = kubeflow.trainer.TrainerClient(backend_config=config)
github_container_registry = (
    "ghcr.io/mxochicale/kubeflowtrainerimage/kubeflowtrainerimage:v0.0.4"
)

# Create environment variables for distributed training
env_vars = {
    "MASTER_ADDR": "localhost",
    "MASTER_PORT": "12355",
    "WORLD_SIZE": "1",
    "RANK": "0",
    "LOCAL_RANK": "0"
}

command = TrainerCommand(command=["./my-entrypoint.sh"])

In [4]:
job_id = trainer.train(
    runtime=trainer.get_runtime("torch-distributed"),
    trainer=CustomTrainerContainer(
        image=github_container_registry,
        env=env_vars        
    ),
    options=[command],
)

In [5]:
#Check job status directly
job = trainer.get_job(job_id)
print(f"\nJob ID: {job_id}")
print(f"Job Status: {job.status}")
print(f"Creation Time: {job.creation_timestamp}")
print(f"\nJob details: {job}")


Job ID: m9af3a576d8c
Job Status: Created
Creation Time: 2026-03-12 11:34:25+00:00

Job details: TrainJob(name='m9af3a576d8c', runtime=Runtime(name='torch-distributed', trainer=RuntimeTrainer(trainer_type=<TrainerType.CUSTOM_TRAINER: 'CustomTrainer'>, framework='torch', image='pytorch/pytorch:2.7.1-cuda12.8-cudnn9-runtime', num_nodes=1, device='Unknown', device_count='Unknown'), pretrained_model=None), steps=[], num_nodes=1, creation_timestamp=datetime.datetime(2026, 3, 12, 11, 34, 25, tzinfo=TzInfo(0)), status='Created')


In [6]:
print("Waiting for job logs...")
wait_count = 0

while True:
    initial_logs = list(trainer.get_job_logs(job_id, follow=False))
    if initial_logs:
        print(f"Logs received after {wait_count} seconds:")
        for log in initial_logs:
            print(f"  {log}")
        break
    
    wait_count += 1
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Waiting... ({wait_count}s)")
    time.sleep(1)


Waiting for job logs...
[11:34:27] Waiting... (1s)
[11:34:29] Waiting... (2s)
[11:34:30] Waiting... (3s)
Logs received after 3 seconds:
  [Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
  PyTorch Distributed Environment
  Using device: cpu
  WORLD_SIZE: 1
  RANK: 0
  LOCAL_RANK: 0


In [7]:
for logline in trainer.get_job_logs(job_id, follow=True):
    print(logline)

[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
PyTorch Distributed Environment
Using device: cpu
WORLD_SIZE: 1
RANK: 0
LOCAL_RANK: 0


## Delete the TrainJob
When TrainJob is finished, you can delete the resource.

In [8]:
trainer.delete_job(job_id)